# Extract, Transform, Load

In the last lesson, we focused on exploratory data analysis. Specifically, we extracted information from our MongoDB database in order to describe some characteristics of the DS Lab applicant pool — country of origin, age, and education level. In this lesson, our goal is to design our experiment, and that means we'll need to go beyond extracting information. We'll also need to make some transformations in our data and then load it back into our database.

In Data Science and Data Engineering, the process of taking data from a source, changing it, and then loading it into a database is called ETL, which is short for extract, transform, load. ETL tends to be more programming-intensive than other data science tasks like visualization, so we'll also spend time in this lesson exploring Python as an object-oriented programming language. Specifically, we'll create our own Python class to contain our ETL processes.

In [ ]:
import random

import pandas as pd
from pymongo import MongoClient
from teaching_tools.ab_test.reset import Reset

r = Reset("ip_of_mongo_device")
r.reset_database()

## Connect

In [ ]:
host = "192.209.170.2"

In [ ]:
## Assign the "ds-applicants" collection in the "wqu-abtest" database to the variable name ds_app.

client = MongoClient(host=host, port=27017)
db = client["wqu-abtest"]
ds_app = db["ds-applicants"]
print("client:", type(client))
print("ds_app:", type(ds_app))

## Extract: Developing the Hypothesis

In [ ]:
## Use the aggregate method to calculate the number of applicants that completed and did not complete the admissions quiz.

# How many applicants complete admissions quiz?

result = ds_app.aggregate(
    [
        {
            "$group": {
               "_id": "$admissionsQuiz",
                "count": {"$count": {}}
            }
        }
    ]
)
for r in result:
    if r["_id"] == "incomplete":
        incomplete = r["count"]
    else:
        complete = r["count"]
    
#incomplete = ...
#complete = ...

print("Completed quiz:", complete)
print("Did not complete quiz:", incomplete)

In [ ]:
## Using your results from the previous task, calculate the proportion of new users who have not completed the admissions quiz.

total = complete + incomplete
prop_incomplete = incomplete/total
print(
    "Proportion of users who don't complete admissions quiz:", round(prop_incomplete, 2)
)

Now that we know that around a quarter of DS Lab applicants don't complete the admissions quiz, is there anything we can do improve the completion rate?

This is a question that we asked ourselves at WQU. In fact, here's a conversation between Nicholas and Anne (Program Director at WQU) where they identify the issue, come up with a hypothesis, and then decide how they'll conduct their experiment.

A hypothesis is an informed guess about what we think is going to happen in an experiment. We probably hope that whatever we're trying out is going to work, but it's important to maintain a healthy degree of skepticism. Science experiments are designed to demonstrate what does work, not what doesn't, so we always start out by assuming that whatever we're about to do won't make a difference (even if we hope it will). The idea that an experimental intervention won't change anything is called a null hypothesis (
), and every experiment either rejects the null hypothesis (meaning the intervention worked), or fails to reject the null hypothesis (meaning it didn't).

The mirror image of the null hypothesis is called an alternate hypothesis (
), and it proceeds from the idea that whatever we're about to do actually will work. If I'm trying to figure out whether exercising is going to help me lose weight, the null hypothesis says that if I exercise, I won't lose any weight. The alternate hypothesis says that if I exercise, I will lose weight.

It's important to keep both types of hypothesis in mind as you work through your experimental design.

In [ ]:
null_hypothesis = """
    There is no relationship between receiving an email and completing the admissions quiz.
    Sending an email to no-quiz applicants does not increase the rate of completion.
"""

alternate_hypothesis ="""
    There is a relationship between receiving an email and completing the admissions quiz.
    Sending an email to no-quiz applicants does increase the rate of completion.
"""

print("Null Hypothesis:", null_hypothesis)
print("Alternate Hypothesis:", alternate_hypothesis)

In [ ]:
## Create a function find_by_date that can search a collection such as "ds-applicants" and return all the no-quiz applicants from a specific date. 

def find_by_date(collection, date_string):
    """Find records in a PyMongo Collection created on a given date.

    Parameters
    ----------
    collection : pymongo.collection.Collection
        Collection in which to search for documents.
    date_string : str
        Date to query. Format must be '%Y-%m-%d', e.g. '2022-06-28'.

    Returns
    -------
    observations : list
        Result of query. List of documents (dictionaries).
    """
    # Convert `date_string` to datetime object
    start = pd.to_datetime(date_string, format="%Y-%m-%d")
    # Offset `start` by 1 day
    end = start + pd.DateOffset(days=1)
    # Create PyMongo query for no-quiz applicants b/t `start` and `end`
    query = {"createdAt": {"$gte": start, "$lt": end}, "admissionsQuiz": "incomplete"}
    # Query collection, get result
    result = collection.find(query)
    # Convert `result` to list
    observations = list(result)
    
    return observations

In [ ]:
## Use your find_by_date function to create a list observations with all the new users created on 2 May 2022.

observations = find_by_date(ds_app, date_string="2022-05-05")

print("observations type:", type(observations))
print("observations len:", len(observations))
observations[0]

# Transform: Designing the Experiment

Okay! Now that we've extracted the data we'll need for the experiment, it's time to get our hands dirty. 

The **transform** stage of ETL involves manipulating the data we just extracted. In this case, we're going to be figuring out which students didn't take the quiz, and assigning them to different experimental groups. To do that, we'll need to *transform* each document in the database by creating a new attribute for each record.

Now we can split the students who didn't take the quiz into two groups: one that will receive a reminder email, and one that will not. Let's make another function that'll do that for us.

Create a function `assign_to_groups` that takes a list of new user documents as input and adds two keys to each document. The first key should be `"inExperiment"`, and its value should always be `True`. The second key should be `"group"`, with half of the records in `"email (treatment)"` and the other half in `"no email (control)"`.

In [ ]:
## Create a function `assign_to_groups` that takes a list of new user documents as input and adds two keys to each document. 
##The first key should be `"inExperiment"`, and its value should always be `True`. The second key should be `"group"`, 
## with half of the records in `"email (treatment)"` and the other half in `"no email (control)"`.

def assign_to_groups(observations):
    """Randomly assigns observations to control and treatment groups.

    Parameters
    ----------
    observations : list or pymongo.cursor.Cursor
        List of users to assign to groups.

    Returns
    -------
    observations : list
        List of documents from `observations` with two additional keys:
        `inExperiment` and `group`.
    """
    # Shuffle `observations`
    random.seed(42)
    random.shuffle(observations)

    # Get index position of item at observations halfway point
    idx = len(observations) // 2

    # Assign first half of observations to control group
    for doc in observations[idx:]:
        doc["inExperiment"] = True
        doc["group"] = "no email (control)"
  

    # Assign second half of observations to treatment group
    for doc in observations[idx:]:
        doc["inExperiment"] = True
        doc["group"] = "email (treatment)"

    return observations


observations_assigned = assign_to_groups(observations)

print("observations_assigned type:", type(observations_assigned))
print("observations_assigned len:", len(observations_assigned))
observations_assigned[0]

In [ ]:
## Define a MongoRepository class with an __init__ method. The __init__ method should accept three arguments: client, db, and collection. 
## Use the docstring below as a guide.

class MongoRepository:
    """Repository class for interacting with MongoDB database.

    Parameters
    ----------
    client : `pymongo.MongoClient`
        By default, `MongoClient(host=host, port=27017)`.
    db : str
        By default, `'wqu-abtest'`.
    collection : str
        By default, `'ds-applicants'`.

    Attributes
    ----------
    collection : pymongo.collection.Collection
        All data will be extracted from and loaded to this collection.
    """

    # Task 7.2.14
    def __init__(
    self,
    client=MongoClient(host="localhost", port= 27017),
    db="wqu-abtest",
    collection="ds-applicants"
    ):
        self.collection = client[db][collection]
   
    # Task 7.2.17
    def find_by_date(self, date_string):
        # Convert `date_string` to datetime object
        start = pd.to_datetime(date_string, format="%Y-%m-%d")
        # Offset `start` by 1 day
        end = start + pd.DateOffset(days=1)
        # Create PyMongo query for no-quiz applicants b/t `start` and `end`
        query = {"createdAt": {"$gte": start, "$lt": end}, "admissionsQuiz": "incomplete"}
        # Query collection, get result
        result = self.collection.find(query)
        # Convert `result` to list
        observations = list(result)
        return observations
    # Task 7.2.18
    
    # Task 7.2.19
   

In [ ]:
##  Create a function export_email that takes a list of documents (like observations_assigned) as input, creates a DataFrame with the emails of all 
## observations in the treatment group, and saves the DataFrame as a CSV file. Then use your function to create a CSV file in the current directory.

def export_treatment_emails(observations_assigned, directory="."):
    """Creates CSV file with email addresses of observations in treatment group.

    CSV file name will include today's date, e.g. `'2022-06-28_ab-test.csv'`,
    and a `'tag'` column where every row will be 'ab-test'.

    Parameters
    ----------
    observations_assigned : list
        Observations with group assignment.
    directory : str, default='.'
        Location for saved CSV file.

    Returns
    -------
    None
    """
    # Put `observations_assigned` docs into DataFrame
    df = pd.DataFrame(observations_assigned)

    # Add `"tag"` column
    df["tag"] = "ab-test"

    # Create mask for treatment group only
    mask = df["group"]  == "email (treatment)"

    # Create filename with date
    date_string = pd.Timestamp.now().strftime(format="%Y-%m-%d")
    filename = directory + "/" + date_string + "_ab-test.csv"


    # Save DataFrame to directory (email and tag only)
    df[mask][["email", "tag"]].to_csv(filename, index=False)


export_treatment_emails(observations_assigned=observations_assigned)

# Load: Preparing the Data

We've *extracted* the data and written a bunch of functions we can use to *transform* the data, so it's time for the third part of this module: *loading* the data.

We've assigned the no-quiz applicants to groups for our experiment, so we should update the records in the `"ds-applicants"` collection to reflect that assignment. Before we update all our records, let's start with just one. 

In [ ]:
## Assign the first item in observations_assigned list to the variable updated_applicant. The assign that applicant's ID to the variable applicant_id.

updated_applicant = observations_assigned[0]
applicant_id = updated_applicant["_id"]
print("applicant type:", type(updated_applicant))
print(updated_applicant)
print()
print("applicant_id type:", type(applicant_id))
print(applicant_id)

In [ ]:
## Use the find_one method together with the applicant_id from the previous task to locate the original record in the "ds-applicants" collection.

# Find original record for `applicant_id`
ds_app.find_one({"_id": applicant_id})

In [ ]:
##  Use the update_one method to update the record with the new information in updated_applicant. Once you're done, rerun your query 
## from the previous task to see if the record has been updated.

result = ds_app.update_one(
    filter = {"_id": applicant_id},
    update = {"$set": updated_applicant}
)
print("result type:", type(result))

Note that when we update the document, we get a result back. Before we update multiple records, let's take a moment to explore what result is — and how it relates to object oriented programming in Python.

In [ ]:
## Use the dir function to inspect result. Once you see some of the attributes, try to access them. For instance, what does the 
## raw_result attribute tell you about the success of your record update?

# Access methods and attributes using `dir`
dir(result)
# Access `raw_result` attribute
result.raw_result

We know how to update a record, and we can interpret our operation results. Since we can do it for one record, we can do it for all of them! So let's update the records for all the observations in our experiment.

In [ ]:
## Create a function update_applicants that takes a list of document like as input, updates the corresponding documents in a collection, 
## and returns a dictionary with the results of the update. Then use your function to update "ds-applicants" with observations_assigned. 

def update_applicants(collection, observations_assigned):
    """Update applicant documents in collection.

    Parameters
    ----------
    collection : pymongo.collection.Collection
        Collection in which documents will be updated.

    observations_assigned : list
        Documents that will be used to update collection

    Returns
    -------
    transaction_result : dict
        Status of update operation, including number of documents
        and number of documents modified.
    """
    # Initialize counters
    n = 0
    n_modified = 0

    # Iterate through applicants
    for doc in observations_assigned:
        # Update doc
        result = collection.update_one(
            filter={"_id": doc["_id"]},
            update={"$set": doc}
        )
        # Update counters
        n += result.matched_count
        n_modified += result.modified_count

    # Create results
    transaction_result = {"n": n, "nModified": n_modified}
   
    return transaction_result


In [ ]:
result = update_applicants(ds_app, observations_assigned)
print("result type:", type(result))
result

Note that if you run the above cell multiple times, the value for `result["nModified"]` will go to `0`. This is because you've already updated the documents. 

In [ ]:
## Create an instance of your MongoRepository and assign it to the variable name repo.

repo = MongoRepository()
print("repo type:", type(repo))
repo

In [ ]:
## Extract the collection attribute from repo, and assign it to the variable c_test. Is the c_test the correct data type?

c_test = repo.collection
print("c_test type:", type(c_test))
c_test

In [ ]:
#Using your function as a model, create a find_by_date method for your MongoRepository class. It should take only one argument: date_string. 
## Once you're done, test your method by extracting all the users who created account on 15 May 2022.

may_15_users = repo.find_by_date(date_string="2022-05-15")
print("may_15_users type", type(may_15_users))
print("may_15_users len", len(may_15_users))
may_15_users[:3]

In [ ]:
## Using your function as a model, create an update_applicants method for your MongoRepository class. It should take one argument: documents. 
## To test your method, use the function to update the documents in observations_assigned.

result = repo.update_applicants(observations_assigned)
print("result type:", type(result))
result

Create an assign_to_groups method for your MongoRepository class. Note that it should work differently than your original function. It will take one argument: date_string. It should find users from that date, assign them to groups, update the database, and return the results of the transaction. Once you're done, use your method to assign all the users who created account on 14 May 2022, to groups.